# Contextual Multi-Armed Bandit

For the contextual multi-armed bandit (sMAB) when user information is available (context), we implemented a generalisation of Thompson sampling algorithm ([Agrawal and Goyal, 2014](https://arxiv.org/pdf/1209.3352.pdf)) based on PyMC3.

![title](img/cmab.png)

The following notebook contains an example of usage of the class Cmab, which implements the algorithm above.

In [1]:
import numpy as np

from pybandits.cmab import CmabBernoulli
from pybandits.model import BayesianLogisticRegression, StudentT

/home/runner/.cache/pypoetry/virtualenvs/pybandits-vYJB-miV-py3.10/lib/python3.10/site-packages/pydantic/_migration.py:283: UserWarning: `pydantic.generics:GenericModel` has been moved to `pydantic.BaseModel`.
  warnings.warn(f'`{import_path}` has been moved to `{new_location}`.')


In [2]:
n_samples = 1000
n_features = 5

First, we need to define the input context matrix $X$ of size ($n\_samples, n\_features$) and the mapping of possible actions $a_i \in A$ to their associated model.

In [3]:
# context
X = 2 * np.random.random_sample((n_samples, n_features)) - 1  # random float in the interval (-1, 1)
print("X: context matrix of shape (n_samples, n_features)")
print(X[:10])

X: context matrix of shape (n_samples, n_features)
[[-0.74465219 -0.53193614  0.12224044 -0.99015673 -0.76294607]
 [-0.62016335 -0.40783978  0.2985217  -0.15183202  0.32562301]
 [-0.92291768  0.53431878 -0.70327439  0.72208905  0.46444789]
 [ 0.9140948   0.95587127 -0.21460164 -0.96328873  0.41088823]
 [-0.26554623  0.4253337   0.94301153 -0.80679341 -0.66928817]
 [ 0.80680645 -0.44555922  0.79170955  0.07494193 -0.30577365]
 [ 0.82771898  0.53267479 -0.87455816 -0.54132801 -0.66074041]
 [-0.92482328  0.42384482 -0.21613338 -0.34342146 -0.74884546]
 [ 0.73971091  0.69507845  0.71346875  0.35909516  0.25966928]
 [-0.62895072  0.13145971  0.30552868 -0.06931829 -0.2342116 ]]


In [4]:
# define action model
actions = {
    "a1": BayesianLogisticRegression(alpha=StudentT(mu=1, sigma=2), betas=n_features * [StudentT()]),
    "a2": BayesianLogisticRegression(alpha=StudentT(mu=1, sigma=2), betas=n_features * [StudentT()]),
}

We can now init the bandit given the mapping of actions $a_i$ to their model.

In [5]:
# init contextual Multi-Armed Bandit model
cmab = CmabBernoulli(actions=actions)

The predict function below returns the action selected by the bandit at time $t$: $a_t = argmax_k P(r=1|\beta_k, x_t)$. The bandit selects one action per each sample of the contect matrix $X$.

In [6]:
# predict action
pred_actions, _, _ = cmab.predict(X)
print("Recommended action: {}".format(pred_actions[:10]))

Recommended action: ['a2', 'a1', 'a2', 'a1', 'a1', 'a1', 'a2', 'a1', 'a2', 'a1']


Now, we observe the rewards from the environment. In this example rewards are randomly simulated. 

In [7]:
# simulate reward from environment
simulated_rewards = np.random.randint(2, size=n_samples)
print("Simulated rewards: {}".format(simulated_rewards[:10]))

Simulated rewards: [0 0 1 0 0 0 0 0 1 1]


Finally, we update the model providing per each action sample: (i) its context $x_t$ (ii) the action $a_t$ selected by the bandit, (iii) the corresponding reward $r_t$.

In [8]:
# update model
cmab.update(X, actions=pred_actions, rewards=simulated_rewards)

ValidationError: 2 validation errors for BaseCmabBernoulli.update
actions
  Got multiple values for argument [type=multiple_argument_values, input_value=['a2', 'a1', 'a2', 'a1', ... 'a2', 'a1', 'a1', 'a2'], input_type=list]
    For further information visit https://errors.pydantic.dev/2.11/v/multiple_argument_values
context
  Missing required argument [type=missing_argument, input_value=ArgsKwargs((CmabBernoulli... 0, 0, 0, 0, 1, 0, 1])}), input_type=ArgsKwargs]
    For further information visit https://errors.pydantic.dev/2.11/v/missing_argument